In [ ]:
# Importamos las librerías necesarias de scikit-learn

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import RepeatedStratifiedKFold

In [ ]:
# 1. Definir características (X) y objetivo (y)
# Suponiendo que tus características predictoras son los dominios más edad y nivel de estudio
# (Asegúrate de que 'age_num' sea el nombre correcto en tu dataframe)
features = DOMINIOS + ["age", "nivel_estudio"]
X = df_complete[features]

# Asegurar que el target sea un valor entero válido
y = pd.to_numeric(df_complete["dc"], errors='coerce').fillna(-1).astype(int)

# Filtramos filas que tengan un target inválido (como 'No determinada' que se volvió -1)
mask = y != -1
X = X[mask]
y = y[mask]


# Definir el diccionario de modelos con balanceo de clases algorítmico
models = {
    "Regresión Logística": LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
    # Limitamos la profundidad del Random Forest a 3 para evitar overfitting en el dataset pequeño
    "Random Forest": RandomForestClassifier(class_weight="balanced", n_estimators=200, min_samples_leaf=5,min_samples_split=10,max_features="sqrt",  max_depth=3, random_state=42),
    "SVM (Lineal)": SVC(kernel="linear", class_weight="balanced", random_state=42),
    "SVM (RBF)": SVC(kernel="rbf", class_weight="balanced", random_state=42)
}

# Configurar la validación cruzada estratificada y las métricas recomendadas para datos desbalanceados
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Métricas recomendadas para datos desbalanceados
scoring = ['f1_macro', 'balanced_accuracy']

# Lista para guardar los resultados
results = []

# Bucle principal de evaluación
print("Iniciando entrenamiento y validación cruzada...")
for name, model in models.items():
    
    # Creamos un Pipeline de 3 pasos:
    # Paso 1: Rellenar NaNs con la mediana (SimpleImputer)
    # Paso 2: Escalar los datos a Media=0, Std=1 (StandardScaler)
    # Paso 3: Entrenar el clasificador
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')), 
        ('scaler', StandardScaler()),                 
        ('classifier', model)
    ])
    
    # Ejecutar la validación cruzada
    cv_scores = cross_validate(pipeline, X, y, cv=cv, scoring=scoring, return_train_score=False)
    
    # Calcular y almacenar las medias y desviaciones estándar
    results.append({
        "Modelo": name,
        "F1-Score Macro": cv_scores['test_f1_macro'].mean(),
        "F1-Score (Std)": cv_scores['test_f1_macro'].std(),
        "Balanced Accuracy": cv_scores['test_balanced_accuracy'].mean(),
        "Bal. Acc. (Std)": cv_scores['test_balanced_accuracy'].std()
    })

# Convertir resultados a DataFrame, ordenar de mejor a peor, y mostrar
df_results = pd.DataFrame(results).sort_values(by="F1-Score Macro", ascending=False)
print("-" * 70)
print("🏆 RESULTADOS DE LOS MODELOS (ORDENADOS POR F1-SCORE MACRO)")
print("-" * 70)
display(df_results.round(4))